<a href="https://colab.research.google.com/github/melissanoarianna1-commits/finpulse-ai/blob/main/notebooks/00_data_collection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================
# RUN THIS FIRST — Colab Setup
# ============================================
!pip install datasets feedparser sentence-transformers -q

# Clone the repo to access project structure
!git clone https://github.com/melissanoarianna1-commits/finpulse-ai.git 2>/dev/null || echo 'Repo already cloned'
import os
os.chdir('/content/finpulse-ai')
os.makedirs('data/raw', exist_ok=True)
os.makedirs('data/processed', exist_ok=True)
os.makedirs('data/validated', exist_ok=True)
os.makedirs('models/sentiment', exist_ok=True)
os.makedirs('models/recommender', exist_ok=True)
os.makedirs('docs', exist_ok=True)
print('✅ Setup complete. Working directory:', os.getcwd())

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 2.9 MB/s eta 0:00:00
✅ Setup complete. Working directory: /content/finpulse-ai


# FinPulse AI — Data Collection & Preparation

This notebook handles **Step 2** of the FinPulse AI pipeline:
1. Download and prepare the **Financial PhraseBank** dataset (for sentiment model)
2. Collect and label **paper abstracts** from arXiv (for recommender model)
3. Run **Great Expectations** validation on both datasets

## Why Data Quality Matters in Financial Risk
In banking, model risk management (Basel III/IV) requires that all model inputs be validated.
Garbage in = garbage out. Our Great Expectations layer enforces this principle programmatically.

## 1. Financial PhraseBank — Sentiment Dataset

The [Financial PhraseBank](https://huggingface.co/datasets/financial_phrasebank) contains ~4,845 English sentences
from financial news, annotated by 5-8 financial domain experts as **positive**, **negative**, or **neutral**.

We use the `sentences_allagree` subset (sentences where all annotators agreed) for highest label quality.
This is the standard benchmark dataset for financial sentiment analysis.

In [6]:
import pandas as pd
import numpy as np
from datasets import load_dataset
from sklearn.model_selection import train_test_split

# Download Financial PhraseBank from HuggingFace
dataset = load_dataset("mteb/financial_phrasebank")
df_sentiment = pd.DataFrame(dataset['train'])

# Map labels: 0=negative, 1=neutral, 2=positive
label_map = {0: 'negative', 1: 'neutral', 2: 'positive'}
df_sentiment['label_text'] = df_sentiment['label'].map(label_map)

print(f'Total sentences: {len(df_sentiment)}')
print(f'\nLabel distribution:')
print(df_sentiment['label_text'].value_counts())
print(f'\nSample entries:')
df_sentiment.head()

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/92.4k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/89.5k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1131 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1128 [00:00<?, ? examples/s]

Total sentences: 1131

Label distribution:
label_text
neutral     694
positive    285
negative    152
Name: count, dtype: int64

Sample entries:


,text,label,label_text
0,Lemminkainen Infra Oy 's subsidiary Lemminkain...,1,neutral
1,"( ADP News ) - Feb 12 , 2009 - Finnish constru...",0,negative
2,In the meantime the CEO 's duties will be assu...,1,neutral
3,"The Company serves approximately 3,000 custome...",1,neutral
4,L+ñnnen Tehtaat 's Food Division was reorganis...,1,neutral


In [7]:
# This dataset already has train/test splits — we use test as validation
train_sent = pd.DataFrame(dataset['train'])
val_sent = pd.DataFrame(dataset['test'])

# Map labels
label_map = {0: 'negative', 1: 'neutral', 2: 'positive'}
train_sent['label_text'] = train_sent['label'].map(label_map)
val_sent['label_text'] = val_sent['label'].map(label_map)

# Rename 'text' to 'sentence' for consistency with the rest of the pipeline
train_sent = train_sent.rename(columns={'text': 'sentence'})
val_sent = val_sent.rename(columns={'text': 'sentence'})

print(f'Training set: {len(train_sent)} sentences')
print(f'Validation set: {len(val_sent)} sentences')
print(f'\nTraining label distribution:')
print(train_sent['label_text'].value_counts(normalize=True).round(3))

# Save
train_sent.to_csv('data/processed/sentiment_train.csv', index=False)
val_sent.to_csv('data/processed/sentiment_val.csv', index=False)
print('\n✅ Sentiment data saved')

Training set: 1131 sentences
Validation set: 1128 sentences

Training label distribution:
label_text
neutral     0.614
positive    0.252
negative    0.134
Name: proportion, dtype: float64

✅ Sentiment data saved


## 2. Paper Abstracts — Recommender Dataset

For the recommender model, we need paper abstracts labeled as **relevant** or **not relevant**
to Banking & Finance research.

**Strategy:**
- **Relevant papers**: Fetch from arXiv categories `q-fin.RM` (Risk Management), `q-fin.GN` (General Finance), `q-fin.CP` (Computational Finance)
- **Not relevant papers**: Fetch from unrelated arXiv categories like `cs.CV` (Computer Vision), `physics.bio-ph` (Biological Physics)

This gives us a clean binary classification task with naturally labeled data.

In [8]:
import feedparser
import time

def fetch_arxiv(query, max_results=200):
    url = f'http://export.arxiv.org/api/query?search_query={query}&start=0&max_results={max_results}&sortBy=submittedDate&sortOrder=descending'
    feed = feedparser.parse(url)
    papers = []
    for entry in feed.entries:
        abstract = entry.summary.replace('\n', ' ').strip()
        if len(abstract) > 50:
            papers.append({
                'title': entry.title.replace('\n', ' ').strip(),
                'abstract': abstract,
                'authors': ', '.join([a.name for a in entry.authors]),
                'date': entry.published[:10],
                'source': 'arXiv',
                'url': entry.link,
            })
    return pd.DataFrame(papers)

print('Fetching relevant papers (finance)...')
df_relevant = fetch_arxiv('cat:q-fin.RM+OR+cat:q-fin.GN+OR+cat:q-fin.CP', 200)
df_relevant['relevant'] = 1
print(f'  → {len(df_relevant)} relevant papers')

time.sleep(3)  # Be polite to arXiv API

print('Fetching non-relevant papers (other fields)...')
df_irrelevant = fetch_arxiv('cat:cs.CV+OR+cat:physics.bio-ph+OR+cat:math.CO', 200)
df_irrelevant['relevant'] = 0
print(f'  → {len(df_irrelevant)} non-relevant papers')

# Combine and shuffle
df_papers = pd.concat([df_relevant, df_irrelevant], ignore_index=True)
df_papers = df_papers.sample(frac=1, random_state=42).reset_index(drop=True)
print(f'\nTotal: {len(df_papers)} papers ({df_papers["relevant"].sum()} relevant, {(df_papers["relevant"]==0).sum()} not relevant)')

Fetching relevant papers (finance)...
  → 200 relevant papers
Fetching non-relevant papers (other fields)...
  → 200 non-relevant papers

Total: 400 papers (200 relevant, 200 not relevant)


In [9]:
# Train/Val split
train_papers, val_papers = train_test_split(
    df_papers, test_size=0.2, random_state=42, stratify=df_papers['relevant']
)

print(f'Training: {len(train_papers)} | Validation: {len(val_papers)}')

# Save
train_papers.to_csv('data/processed/papers_train.csv', index=False)
val_papers.to_csv('data/processed/papers_val.csv', index=False)
df_papers.to_csv('data/processed/paper_relevance.csv', index=False)
print('✅ Paper data saved to data/processed/')

Training: 320 | Validation: 80
✅ Paper data saved to data/processed/


## 3. Data Validation

We validate both datasets to ensure quality before model training.
This mirrors how banks validate data inputs for risk models under Basel III/IV.

In [10]:
# ---- Validate Sentiment Dataset ----
print('='*60)
print('VALIDATING: Financial PhraseBank (Sentiment Dataset)')
print('='*60)

checks = {
    'no_null_sentences': train_sent['sentence'].notna().all(),
    'no_null_labels': train_sent['label'].notna().all(),
    'valid_labels': set(train_sent['label'].unique()).issubset({0, 1, 2}),
    'no_empty_sentences': (train_sent['sentence'].str.len() > 0).all(),
    'min_sentence_length': (train_sent['sentence'].str.len() >= 10).all(),
    'no_duplicates': not train_sent.duplicated(subset=['sentence']).any(),
    'all_classes_present': len(train_sent['label'].unique()) == 3,
    'class_balance_ok': train_sent['label'].value_counts(normalize=True).min() > 0.05,
}

for name, passed in checks.items():
    print(f'  {"✅ PASS" if passed else "❌ FAIL"} | {name}')
print(f'\nOverall: {"✅ ALL PASSED" if all(checks.values()) else "❌ SOME FAILED"}')

VALIDATING: Financial PhraseBank (Sentiment Dataset)
  ✅ PASS | no_null_sentences
  ✅ PASS | no_null_labels
  ✅ PASS | valid_labels
  ✅ PASS | no_empty_sentences
  ❌ FAIL | min_sentence_length
  ✅ PASS | no_duplicates
  ✅ PASS | all_classes_present
  ✅ PASS | class_balance_ok

Overall: ❌ SOME FAILED


In [11]:
# ---- Validate Paper Dataset ----
print('='*60)
print('VALIDATING: Paper Abstracts (Recommender Dataset)')
print('='*60)

checks_p = {
    'no_null_titles': train_papers['title'].notna().all(),
    'no_null_abstracts': train_papers['abstract'].notna().all(),
    'no_null_authors': train_papers['authors'].notna().all(),
    'no_null_dates': train_papers['date'].notna().all(),
    'abstract_min_length': (train_papers['abstract'].str.len() >= 50).all(),
    'no_duplicate_titles': not train_papers.duplicated(subset=['title']).any(),
    'both_classes_present': len(train_papers['relevant'].unique()) == 2,
    'class_balance_ok': train_papers['relevant'].value_counts(normalize=True).min() > 0.3,
}

for name, passed in checks_p.items():
    print(f'  {"✅ PASS" if passed else "❌ FAIL"} | {name}')
print(f'\nOverall: {"✅ ALL PASSED" if all(checks_p.values()) else "❌ SOME FAILED"}')

VALIDATING: Paper Abstracts (Recommender Dataset)
  ✅ PASS | no_null_titles
  ✅ PASS | no_null_abstracts
  ✅ PASS | no_null_authors
  ✅ PASS | no_null_dates
  ✅ PASS | abstract_min_length
  ✅ PASS | no_duplicate_titles
  ✅ PASS | both_classes_present
  ✅ PASS | class_balance_ok

Overall: ✅ ALL PASSED


In [12]:
# Save validated data
if all(checks.values()):
    train_sent.to_csv('data/validated/sentiment_train.csv', index=False)
    val_sent.to_csv('data/validated/sentiment_val.csv', index=False)
    print('✅ Sentiment data validated and saved')

if all(checks_p.values()):
    train_papers.to_csv('data/validated/papers_train.csv', index=False)
    val_papers.to_csv('data/validated/papers_val.csv', index=False)
    print('✅ Paper data validated and saved')

print('\n--- Step 2 Complete ---')
print('Next: Run 01_sentiment_model.ipynb')

✅ Paper data validated and saved

--- Step 2 Complete ---
Next: Run 01_sentiment_model.ipynb
